In [1]:
import json
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from IPython.display import Markdown, display

from pathlib import Path
import pandas as pd
import numpy as np

In [2]:
class ExperimentMetrics:
    def __init__(self, json_file):
        with open(json_file, "r") as f:
            self.results = json.load(f)

        # Convert arrays (as you already do)
        self.results['selected_model_predict_proba_array'] = np.array(
            self.results['selected_model_predict_proba_array']
        )
        self.results['selected_model_predict_array'] = np.array(
            self.results['selected_model_predict_array']
        )
        self.results['shap_values'] = np.array(self.results['shap_values'])
        self.results['explain_tabnet_matrix'] = np.array(
            self.results['explain_tabnet_matrix']
        )

        self.n = self.results['len_x_test']
        self.feature_names = self.results['feature_names']

    def fidelity_probability(self):
        lime = [
            1 - abs(
                self.results['selected_model_predict_proba_array'][i, 1]
                - self.results['lime_explanations_list_local_pred'][i]
            )
            for i in range(self.n)
        ]

        shap = [
            1 - abs(
                self.results['selected_model_predict_proba_array'][i, 1]
                - (
                    self.results['exp_shap_expected_value'][1]
                    + self.results['shap_values'][i][:, 1].sum()
                )
            )
            for i in range(self.n)
        ]

        return {
            "LIME": (np.mean(lime), np.std(lime)),
            "SHAP": (np.mean(shap), np.std(shap)),
        }

    def fidelity_decision(self):
        lime_pred = np.array(
            [int(v > 0.5) for v in self.results['lime_explanations_list_local_pred']]
        )

        shap_pred = np.array([
            int(
                self.results['exp_shap_expected_value'][1]
                + self.results['shap_values'][i][:, 1].sum() > 0.5
            )
            for i in range(self.n)
        ])

        model_pred = self.results['selected_model_predict_array']

        kronecker = lambda x, y: [1 if a == b else 0 for a, b in zip(x, y)]

        lime_fd = kronecker(lime_pred, model_pred)
        shap_fd = kronecker(shap_pred, model_pred)

        return {
            "LIME": (np.mean(lime_fd), np.std(lime_fd)),
            "SHAP": (np.mean(shap_fd), np.std(shap_fd)),
        }

    @staticmethod
    def _calculate_simplicity(values, threshold):
        scores = []
        for v in values:
            abs_v = np.abs(v)
            scores.append(np.sum(abs_v > max(abs_v) * threshold))
        return scores


    def simplicity(self, thresholds=[0.1, 0.05, 0.01]):
        lime_vals = [
            list(zip(*self.results['lime_explanations_list__'][i]))[1]
            for i in range(self.n)
        ]

        methods = {
            "LIME": lime_vals,
            "SHAP": self.results['shap_values'][:, :, 0],
            "EBM": self.results['ebm_local_scores'],
            "TabNet": self.results['explain_tabnet_matrix'],
        }

        mean_rows, std_rows = [], []

        for name, values in methods.items():
            mean_row = {"Method": name}
            std_row = {"Method": name}

            for t in thresholds:
                s = self._calculate_simplicity(values, t)
                mean_row[t] = np.mean(s)
                std_row[t] = np.std(s)

            mean_rows.append(mean_row)
            std_rows.append(std_row)

        return (
            pd.DataFrame(mean_rows).set_index("Method"),
            pd.DataFrame(std_rows).set_index("Method"),
        )

    @staticmethod
    def _retain_top(features, n):
        arr = np.abs(features)
        idx = np.argpartition(arr, -n)[-n:]
        mask = np.zeros_like(arr, dtype=bool)
        mask[idx] = True
        return np.where(mask, features, 0)


    def _lime_importance(self, lime_exp):
        feature_importance = {}
        for feature, sc in lime_exp:
            if feature.find('< ') > -1:
                feature = feature.split(' < ')[1]
            feature = feature.split(' > ')[0].strip()
            feature = feature.split(' <= ')[0].strip()
            for f in ['>', '<', '=']:
                if feature.find(f) > -1:
                    print('error', feature)
            feature_importance[feature] = sc
        return np.array([feature_importance.get(feature, 0) for feature in self.feature_names])

    def consistency(self):
        shap = lambda i, n: self._retain_top(self.results['shap_values'][i][:, 1], n)
        lime = lambda i, n: self._retain_top(
            self._lime_importance(self.results['lime_explanations_list__'][i]), n
        )
        tab = lambda i, n: self._retain_top(self.results['explain_tabnet_matrix'][i], n)
    
        def get_consistency(method1, method2):
            per_instance_scores = []
            for i in range(self.n):
                corr_over_n = np.array([
                    spearmanr(method1(i, n), method2(i, n))[0]
                    for n in range(1, 6)
                ], dtype=float)
    
                per_instance_scores.append(np.nanmean(corr_over_n))  # avg over top-N
            return np.array(per_instance_scores, dtype=float)
    
        lime_shap = get_consistency(lime, shap)
        lime_tab = get_consistency(lime, tab)
        tab_shap = get_consistency(tab, shap)
    
        return {
            "LIME–SHAP": (float(np.nanmean(lime_shap)), float(np.nanstd(lime_shap, ddof=1))),
            "LIME–TabNet": (float(np.nanmean(lime_tab)), float(np.nanstd(lime_tab, ddof=1))),
            "TabNet–SHAP": (float(np.nanmean(tab_shap)), float(np.nanstd(tab_shap, ddof=1))),
        }

    @staticmethod
    def _robustness_array(original, noisy):
        """Vector robustness for SHAP/EBM/TabNet-like arrays."""
        org = np.array(original)
        noi = np.array(noisy)
        diff = np.abs(org - noi)

        divisor = max(org.max(), noi.max()) - min(org.min(), noi.min())
        if divisor == 0:
            return diff  # avoid divide-by-zero; change=0 in degenerate case

        return (diff/divisor)

    def _lime_robustness(self):
        """Robustness for LIME where explanations are lists of (feature, score)."""
        original = self.results['lime_explanations_list__']
        noisy = self.results['lime_explanations_list_noisy__']

        diff_list = []
        _max, _min = 0.0, 0.0

        for i in range(len(original)):
            d1, d2 = dict(original[i]), dict(noisy[i])

            keys = set(d1.keys()) | set(d2.keys())
            v1 = np.array([d1.get(k, 0.0) for k in keys], dtype=float)
            v2 = np.array([d2.get(k, 0.0) for k in keys], dtype=float)

            _max = max(_max, v1.max(), v2.max())
            _min = min(_min, v1.min(), v2.min())

            diff_list.append(np.abs(v1 - v2).mean())

        divisor = _max - _min
        if divisor == 0:
            return np.array(diff_list) # degenerate fallback
        return np.array(diff_list)/divisor

    def robustness(self):
        """Returns per-method robustness scores (scalars)."""
        return {
            "LIME": self._lime_robustness(),
            "SHAP": self._robustness_array(
                self.results['shap_values'],
                self.results['shap_values_noisy']
            ),
            "EBM": self._robustness_array(
                self.results['ebm_local_scores'],
                self.results['ebm_local_scores_noisy']
            ),
            "TabNet": self._robustness_array(
                self.results['explain_tabnet_matrix'],
                self.results['explain_tabnet_matrix_noisy']
            ),
        }

    def anchors(self):
        rows = []

        for t, arr in self.results['anchors_metrics_dict'].items():
            mean = np.mean(arr, axis=0)
            std = np.std(arr, axis=0)

            rows.append({
                "Threshold": t,
                "Precision_mean": mean[0],
                "Coverage_mean": mean[1],
                "Simplicity_mean": mean[2],
                "Robustness_mean": mean[3],
                "Precision_std": std[0],
                "Coverage_std": std[1],
                "Simplicity_std": std[2],
                "Robustness_std": std[3],
            })

        return pd.DataFrame(rows).set_index("Threshold")

In [3]:
class MetricsDisplay:

    @staticmethod
    def section(title):
        display(Markdown(f"### {title}"))

    @staticmethod
    def dataframe(df, precision=3):
        display(df.round(precision))

    @staticmethod
    def mean_std_dict(d, precision=4):
        rows = []
        for k, (m, s) in d.items():
            rows.append({
                "Method": k,
                "Mean": round(m, precision),
                "Std": round(s, precision)
            })
        df = pd.DataFrame(rows).set_index("Method")
        display(df)


In [4]:
def run_all_datasets(dataset_files):
    """
    dataset_files: dict like {"diaretino": "output/diaretino.csv_out.json", ...}

    Returns:
      metrics: dict dataset -> ExperimentMetrics instance
    """
    metrics = {ds: ExperimentMetrics(path) for ds, path in dataset_files.items()}
    return metrics


In [5]:
dataset_files = {
    "PDHS (C-section)": "output/cSection_train_test.csvs_out.json",
    "Diabetes retinopathy": "output/diaretino.csv_out.json",
    "Breast cancer": "output/breast_cancer.df_out.json",
    "ESDRPD": "output/Dataset 2 _ Early-stage diabetes risk prediction dataset (ESDRPD).xlsx_out.json",
}

all_exp = run_all_datasets(dataset_files)

In [6]:
def mean_pm_std(mean, std, decimals=2):
    if np.isnan(std):
        return f"{mean:.{decimals}f}"
    return f"{mean:.{decimals}f} ± {std:.{decimals}f}"


In [7]:
def table_fidelity_combined(all_exp, decimals=4):
    """
    Combined fidelity table:
    - Probability-based fidelity
    - Decision-based fidelity
    """
    rows = []

    for ds, exp in all_exp.items():
        fid_prob = exp.fidelity_probability()
        fid_dec = exp.fidelity_decision()

        rows.append({
            "Dataset": ds,
            "LIME (Score)": mean_pm_std(*fid_prob["LIME"], decimals),
            "SHAP (Score)": mean_pm_std(*fid_prob["SHAP"], decimals),
            "LIME (Decision)": mean_pm_std(*fid_dec["LIME"], decimals),
            "SHAP (Decision)": mean_pm_std(*fid_dec["SHAP"], decimals),
        })

    return pd.DataFrame(rows).set_index("Dataset")


In [30]:
# def table_simplicity(all_exp, chosen_threshold=0.05, decimals=2):
#     rows = []
#     for ds, exp in all_exp.items():
#         mean_df, std_df = exp.simplicity()
#         row = {"Dataset": ds}
#         for method in mean_df.index:
#             m = float(mean_df.loc[method, chosen_threshold])
#             s = float(std_df.loc[method, chosen_threshold])
#             row[method] = mean_pm_std(m, s, decimals=decimals)
#         rows.append(row)
#     df = pd.DataFrame(rows).set_index("Dataset")
#     # Order columns nicely
#     ordered = [c for c in ["LIME", "SHAP", "EBM", "TabNet"] if c in df.columns]
#     return df[ordered]

# def table_simplicity_all_thresholds(all_exp, thresholds=(0.1, 0.05, 0.01), decimals=2):
#     rows = []
#     for ds, exp in all_exp.items():
#         mean_df, std_df = exp.simplicity(list(thresholds))
#         row = {"Dataset": ds}

#         for t in thresholds:
#             for method in mean_df.index:
#                 m = float(mean_df.loc[method, t])
#                 s = float(std_df.loc[method, t])
#                 row[f"{t} {method}"] = mean_pm_std(m, s, decimals=decimals)

#         rows.append(row)

#     df = pd.DataFrame(rows).set_index("Dataset")

#     # Optional: make column order nice: by threshold then method
#     ordered_cols = []
#     for t in thresholds:
#         for method in ["LIME", "SHAP", "EBM", "TabNet"]:
#             col = f"{t} {method}"
#             if col in df.columns:
#                 ordered_cols.append(col)

#     return df[ordered_cols]

def table_simplicity_block(all_exp, dataset_order=None,
                           thresholds=(0.1, 0.05, 0.01),
                           method_order=("LIME", "SHAP", "EBM", "TabNet"),
                           decimals=2):

    if dataset_order is None:
        dataset_order = list(all_exp.keys())

    rows = []
    idx = []

    for t in thresholds:
        # Add a "header row" for the threshold (blank values)
        idx.append(f"Threshold={t}")
        rows.append({ds: "" for ds in dataset_order})

        # Add method rows under this threshold
        for method in method_order:
            idx.append(method)
            row = {}
            for ds in dataset_order:
                exp = all_exp[ds]
                mean_df, std_df = exp.simplicity(list(thresholds))

                lookup_method = method
                m = float(mean_df.loc[lookup_method, t])
                s = float(std_df.loc[lookup_method, t])

                row[ds] = f"{m:.{decimals}f} ± {s:.{decimals}f}"
            rows.append(row)

    df = pd.DataFrame(rows, index=idx)
    df.index.name = "Method"
    return df


In [8]:
def table_consistency(all_exp, decimals=2):
    rows = []
    for ds, exp in all_exp.items():
        d = exp.consistency()  # {"LIME–SHAP": (m,s), ...}
        row = {"Dataset": ds}
        for pair, (m, s) in d.items():
            row[pair] = mean_pm_std(m, s, decimals=decimals)
        rows.append(row)
    return pd.DataFrame(rows).set_index("Dataset")


In [57]:
# def table_robustness(all_exp, decimals=4):
#     rows = []
#     for ds, exp in all_exp.items():
#         d = exp.robustness()  # {"LIME": array, "SHAP": array, ...}
#         row = {"Dataset": ds}
#         for method, arr in d.items():
#             arr = np.array(arr, dtype=float)
#             row[method] = mean_pm_std(np.nanmean(arr), np.nanstd(arr, ddof=1), decimals=decimals)
#         rows.append(row)
#     return pd.DataFrame(rows).set_index("Dataset")

def table_robustness_transposed(all_exp, decimals=4):
    methods = ["LIME", "SHAP", "EBM", "TabNet"]
    data = {method: {} for method in methods}

    for ds, exp in all_exp.items():
        rob = exp.robustness()  # {"LIME": array, ...}

        for method in methods:
            arr = np.array(rob[method], dtype=float)
            data[method][ds] = mean_pm_std(
                np.nanmean(arr),
                np.nanstd(arr, ddof=1),
                decimals=decimals
            )

    return pd.DataFrame(data).T

In [33]:
# def table_anchors(all_exp, decimals=2):
#     frames = []
#     for ds, exp in all_exp.items():
#         df = exp.anchors().copy()
#         df.insert(0, "Dataset", ds)
#         frames.append(df.reset_index())  # threshold becomes a column
#     out = pd.concat(frames, ignore_index=True)
#     # Round numeric columns for readability
#     num_cols = [c for c in out.columns if c not in ("Dataset", "Threshold")]
#     out[num_cols] = out[num_cols].round(decimals)
#     return out.set_index(["Dataset", "Threshold"])

def table_anchors(all_exp, decimals=2):
    """
    Anchors table with mean ± std per metric, per dataset, per threshold.
    """
    frames = []

    for ds, exp in all_exp.items():
        df = exp.anchors().copy()
        df.insert(0, "Dataset", ds)

        # Build mean ± std columns
        out = pd.DataFrame({
            "Precision": df.apply(
                lambda r: f"{r['Precision_mean']:.{decimals}f} ± {r['Precision_std']:.{decimals}f}", axis=1
            ),
            "Coverage": df.apply(
                lambda r: f"{r['Coverage_mean']:.{decimals}f} ± {r['Coverage_std']:.{decimals}f}", axis=1
            ),
            "Simplicity": df.apply(
                lambda r: f"{r['Simplicity_mean']:.{decimals}f} ± {r['Simplicity_std']:.{decimals}f}", axis=1
            ),
            "Robustness": df.apply(
                lambda r: f"{r['Robustness_mean']:.{decimals}f} ± {r['Robustness_std']:.{decimals}f}", axis=1
            ),
        }, index=df.index)

        out.insert(0, "Dataset", ds)
        out.insert(1, "Threshold", df.index.astype(str))

        frames.append(out.reset_index(drop=True))

    final = pd.concat(frames, ignore_index=True)
    return final.set_index(["Dataset", "Threshold"])



In [58]:
disp = MetricsDisplay()

disp.section("Table 4: Fidelity (Score & Decision)")
disp.dataframe(table_fidelity_combined(all_exp, decimals=3))

# disp.section("Table 5: Simplicity (threshold = 0.05)")
# disp.dataframe(table_simplicity(all_exp, chosen_threshold=0.05), precision=2)

# disp.section("Table 5: Simplicity across thresholds (Mean ± Std)")
# disp.dataframe(table_simplicity_all_thresholds(all_exp), precision=2)

disp.section("Table 5: Simplicity across thresholds")
simp_tab = table_simplicity_block(all_exp)
display(simp_tab)

disp.section("Table 6: Consistency")
disp.dataframe(table_consistency(all_exp, decimals=2))

# disp.section("Table 7: Robustness")
# disp.dataframe(table_robustness(all_exp, decimals=4))

disp.section("Table 7: Robustness")
disp.dataframe(table_robustness_transposed(all_exp, decimals=4))

# disp.section("Table 8: Anchors")
# disp.dataframe(table_anchors(all_exp), precision=2)

disp.section("Table 8: Anchors")
disp.dataframe(table_anchors(all_exp))

### Table 4: Fidelity (Score & Decision)

,LIME (Score),SHAP (Score),LIME (Decision),SHAP (Decision)
Dataset,,,,
PDHS (C-section),0.890 ± 0.152,1.000 ± 0.000,0.932 ± 0.251,1.000 ± 0.016
Diabetes retinopathy,0.832 ± 0.118,1.000 ± 0.000,0.538 ± 0.499,0.994 ± 0.076
Breast cancer,0.883 ± 0.076,1.000 ± 0.000,0.982 ± 0.131,1.000 ± 0.000
ESDRPD,0.791 ± 0.145,1.000 ± 0.000,0.917 ± 0.276,1.000 ± 0.000


### Table 5: Simplicity across thresholds

,PDHS (C-section),Diabetes retinopathy,Breast cancer,ESDRPD
Method,,,,
Threshold=0.1,,,,
LIME,1.82 ± 0.39,9.86 ± 0.42,9.93 ± 0.37,9.95 ± 0.22
SHAP,3.86 ± 2.82,11.44 ± 2.52,13.86 ± 2.44,8.93 ± 1.99
EBM,7.68 ± 3.93,20.11 ± 5.95,28.40 ± 5.37,20.53 ± 2.12
TabNet,4.31 ± 1.38,6.10 ± 7.07,6.21 ± 9.56,11.53 ± 6.46
Threshold=0.05,,,,
LIME,3.27 ± 0.79,10.00 ± 0.05,10.00 ± 0.00,10.00 ± 0.00
SHAP,8.59 ± 3.37,14.23 ± 2.06,17.42 ± 2.83,12.28 ± 1.46
EBM,16.26 ± 7.08,27.16 ± 4.55,43.55 ± 5.20,27.74 ± 1.92


### Table 6: Consistency

,LIME–SHAP,LIME–TabNet,TabNet–SHAP
Dataset,,,
PDHS (C-section),0.73 ± 0.11,-0.06 ± 0.41,-0.04 ± 0.42
Diabetes retinopathy,0.51 ± 0.27,-0.04 ± 0.19,-0.08 ± 0.21
Breast cancer,0.58 ± 0.21,-0.05 ± 0.22,-0.06 ± 0.20
ESDRPD,0.84 ± 0.14,-0.02 ± 0.13,-0.03 ± 0.11


### Table 7: Robustness

,PDHS (C-section),Diabetes retinopathy,Breast cancer,ESDRPD
LIME,0.0477 ± 0.0338,0.0428 ± 0.0197,0.0422 ± 0.0202,0.0672 ± 0.0272
SHAP,0.0002 ± 0.0004,0.0132 ± 0.0219,0.0078 ± 0.0193,0.0002 ± 0.0009
EBM,0.0000 ± 0.0010,0.0021 ± 0.0103,0.0175 ± 0.0388,0.0000 ± 0.0000
TabNet,0.0006 ± 0.0040,0.0009 ± 0.0106,0.0000 ± 0.0007,0.0201 ± 0.0343


### Table 8: Anchors

Precision     Coverage     Simplicity  \
Dataset              Threshold                                            
PDHS (C-section)     0.9        0.96 ± 0.04  0.29 ± 0.14    2.35 ± 4.05   
                     0.95       0.97 ± 0.04  0.24 ± 0.17    2.81 ± 4.68   
                     0.99       0.99 ± 0.04  0.19 ± 0.19    3.65 ± 5.13   
Diabetes retinopathy 0.9        0.91 ± 0.04  0.02 ± 0.03   11.01 ± 9.38   
                     0.95       0.94 ± 0.05  0.01 ± 0.01  13.89 ± 10.41   
                     0.99       0.96 ± 0.06  0.01 ± 0.01  16.89 ± 10.79   
Breast cancer        0.9        0.99 ± 0.02  0.22 ± 0.11    2.03 ± 0.17   
                     0.95       0.99 ± 0.01  0.22 ± 0.09    2.05 ± 0.25   
                     0.99       1.00 ± 0.00  0.18 ± 0.08    2.37 ± 0.57   
ESDRPD               0.9        0.93 ± 0.03  0.36 ± 0.14    2.12 ± 1.19   
                     0.95       0.99 ± 0.02  0.21 ± 0.10    3.22 ± 1.91   
                     0.99       1.00 ± 0.01  0.19 ± 0.11    3.60 ± 2.04   

                                 Robustness  
Dataset              Threshold               
PDHS (C-section)     0.9        0.39 ± 0.41  
                     0.95       0.40 ± 0.41  
                     0.99       0.47 ± 0.38  
Diabetes retinopathy 0.9        0.54 ± 0.24  
                     0.95       0.56 ± 0.20  
                     0.99       0.56 ± 0.18  
Breast cancer        0.9        0.72 ± 0.18  
                     0.95       0.72 ± 0.21  
                     0.99       0.70 ± 0.22  
ESDRPD               0.9        0.33 ± 0.38  
                     0.95       0.45 ± 0.34  
                     0.99       0.47 ± 0.34

In [62]:
# disp.section("Table 4: Fidelity (Score & Decision)")
# print(
#     table_fidelity_combined(all_exp).to_latex(
#         escape=False,          # keeps ±
#         column_format="lcccc",
#         caption="Fidelity of LIME and SHAP Explanations.",
#         label="tab:fidelity"
#     )
# )

# disp.section("Table 5: Simplicity across thresholds")
# print(
#     simp_tab.to_latex(
#         escape=False,   # keeps ±
#         column_format="lcccc",
#         caption="Simplicity of XAI Methods (Average Number of Features)",
#         label="tab:simplicity",
#         index=True
#     )
# )

# disp.section("Table 6: Consistency")
# print(
#     table_consistency(all_exp).to_latex(
#         escape=False,   # escape=False keeps ±
#         caption="Consistency Between XAI Methods",
#         label="tab:consistency"
#     )
# )  

# disp.section("Table 7: Robustness")
# print(
#     table_robustness_transposed(all_exp).to_latex(
#         escape=False,   # escape=False keeps ±
#         caption="Robustness of XAI Methods",
#         label="tab:robustness"
#     )
# )

disp.section("Table 8: Anchors")
print(
    table_anchors(all_exp).to_latex(
        escape=False,   # keeps ±
        caption="Precision, Coverage, Simplicity, and Robustness of Anchors Across Datasets",
        label="tab:pre_and_cov",
        multirow=True
    )
)

### Table 8: Anchors

\begin{table}
\centering
\caption{Precision, Coverage, Simplicity, and Robustness of Anchors Across Datasets}
\label{tab:pre_and_cov}
\begin{tabular}{llllll}
\toprule
       &      &    Precision &     Coverage &     Simplicity &   Robustness \\
Dataset & Threshold &              &              &                &              \\
\midrule
\multirow{3}{*}{PDHS (C-section)} & 0.9 &  0.96 ± 0.04 &  0.29 ± 0.14 &    2.35 ± 4.05 &  0.39 ± 0.41 \\
       & 0.95 &  0.97 ± 0.04 &  0.24 ± 0.17 &    2.81 ± 4.68 &  0.40 ± 0.41 \\
       & 0.99 &  0.99 ± 0.04 &  0.19 ± 0.19 &    3.65 ± 5.13 &  0.47 ± 0.38 \\
\cline{1-6}
\multirow{3}{*}{Diabetes retinopathy} & 0.9 &  0.91 ± 0.04 &  0.02 ± 0.03 &   11.01 ± 9.38 &  0.54 ± 0.24 \\
       & 0.95 &  0.94 ± 0.05 &  0.01 ± 0.01 &  13.89 ± 10.41 &  0.56 ± 0.20 \\
       & 0.99 &  0.96 ± 0.06 &  0.01 ± 0.01 &  16.89 ± 10.79 &  0.56 ± 0.18 \\
\cline{1-6}
\multirow{3}{*}{Breast cancer} & 0.9 &  0.99 ± 0.02 &  0.22 ± 0.11 &    2.03 ± 0.17 &  0.72 ± 0.18 \\
    

/tmp/ipykernel_2352852/2776618730.py:42: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  table_anchors(all_exp).to_latex(


# Not Needed after this (just for testing the code)

In [13]:
file = 'diaretino.csv'
file = 'cSection_train_test.csvs'
file = 'breast_cancer.df'
# file = 'Dataset 2 _ Early-stage diabetes risk prediction dataset (ESDRPD).xlsx'
exp = ExperimentMetrics('output/'+file+'_out.json')

In [14]:
# print(exp.fidelity_probability())
# print(exp.fidelity_decision())

# simp_mean, simp_std = exp.simplicity()

In [15]:
from IPython.display import display, Latex
import pandas as pd

In [16]:
class MetricsDisplay:
    """Very simple display helpers (no pandas styling, no Jinja2)."""

    @staticmethod
    def section(title):
        display(Markdown(f"### {title}"))

    @staticmethod
    def dataframe(df, precision=3):
        display(df.round(precision))

    @staticmethod
    def mean_std_dict(d, precision=4):
        rows = []
        for k, (m, s) in d.items():
            rows.append({
                "Method": k,
                "Mean": round(m, precision),
                "Std": round(s, precision)
            })
        df = pd.DataFrame(rows).set_index("Method")
        display(df)


In [17]:
def consistency_table(consistency_dict):
    """
    consistency_dict is now: {pair: (mean, std)}
    """
    rows = []
    for pair, (mean, std) in consistency_dict.items():
        rows.append({
            "Method Pair": pair,
            "Mean": mean,
            "Std": std
        })
    return pd.DataFrame(rows).set_index("Method Pair")

In [18]:
def robustness_table(robustness_dict):
    """
    Convert robustness output into a neat summary table
    (mean and std per method).
    """
    rows = []
    for method, values in robustness_dict.items():
        # values may already be a scalar (depending on implementation)
        if hasattr(values, "__len__"):
            mean = np.mean(values)
            std = np.std(values)
        else:
            mean = values
            std = np.nan

        rows.append({
            "Method": method,
            "Mean": mean,
            "Std": std
        })

    return pd.DataFrame(rows).set_index("Method")

In [19]:
disp = MetricsDisplay()

disp.section("Fidelity (Probability)")
disp.mean_std_dict(exp.fidelity_probability())

disp.section("Fidelity (Decision)")
disp.mean_std_dict(exp.fidelity_decision())

### Fidelity (Probability)

,Mean,Std
Method,,
LIME,0.8828,0.0757
SHAP,1.0000,0.0000


### Fidelity (Decision)

,Mean,Std
Method,,
LIME,0.9825,0.1313
SHAP,1.0000,0.0000


In [20]:
simp_mean, simp_std = exp.simplicity()

disp.section("Simplicity — Mean")
disp.dataframe(simp_mean)

disp.section("Simplicity — Std")
disp.dataframe(simp_std)

disp.section("Consistency")
disp.dataframe(consistency_table(exp.consistency()), precision=2)

disp.section("Robustness")
disp.dataframe(robustness_table(exp.robustness()), precision=3)

disp.section("Anchors")
disp.dataframe(exp.anchors(), precision=2)

### Simplicity — Mean

,0.1,0.05,0.01
Method,,,
LIME,9.930,10.000,10.000
SHAP,13.860,17.421,25.462
EBM,28.404,43.550,56.076
TabNet,6.211,6.690,7.368


### Simplicity — Std

,0.1,0.05,0.01
Method,,,
LIME,0.368,0.000,0.000
SHAP,2.441,2.826,2.340
EBM,5.371,5.197,1.556
TabNet,9.559,9.734,9.842


### Consistency

,Mean,Std
Method Pair,,
LIME–SHAP,0.58,0.21
LIME–TabNet,-0.05,0.22
TabNet–SHAP,-0.06,0.20


### Robustness

,Mean,Std
Method,,
LIME,0.042,0.020
SHAP,0.008,0.019
EBM,0.018,0.039
TabNet,0.000,0.001


### Anchors

,Precision_mean,Coverage_mean,Simplicity_mean,Robustness_mean,Precision_std,Coverage_std,Simplicity_std,Robustness_std
Threshold,,,,,,,,
0.9,0.99,0.22,2.03,0.72,0.02,0.11,0.17,0.18
0.95,0.99,0.22,2.05,0.72,0.01,0.09,0.25,0.21
0.99,1.00,0.18,2.37,0.70,0.00,0.08,0.57,0.22


In [21]:
import numpy as np
import pandas as pd

In [22]:
file = 'diaretino.csv'
file = 'cSection_train_test.csvs'
file = 'breast_cancer.df'
# file = 'Dataset 2 _ Early-stage diabetes risk prediction dataset (ESDRPD).xlsx'

json_output_file = 'output/'+file+'_out.json'

In [23]:
with open(json_output_file, 'r') as f:
    results = json.load(f)

results['selected_model_predict_proba_array'] = np.array(results['selected_model_predict_proba_array'])
results['selected_model_predict_array'] = np.array(results['selected_model_predict_array'])
results['shap_values'] = np.array(results['shap_values'])

results['explain_tabnet_matrix'] = np.array(results['explain_tabnet_matrix'])

In [24]:
lime_fidelity_score_list = [1-np.abs(results['selected_model_predict_proba_array'][i, 1] - (results['lime_explanations_list_local_pred'][i])) for i in range(results['len_x_test'])]
print(f'LIME Mean Fidelity_Score: {np.mean(lime_fidelity_score_list):.4f}')
print(f'LIME Std Fidelity_Score: {np.std(lime_fidelity_score_list):.4f}')

# For SHAP, fidelity is usually high as it's designed to be an accurate representation of the model
shap_fidelity_score_list = [1-np.abs(results['selected_model_predict_proba_array'][i, 1] - (results['exp_shap_expected_value'][1] + np.sum(results['shap_values'][i][:, 1]))) for i in range(results['len_x_test'])]
print('')
print(f'SHAP Mean Fidelity_Score: {np.mean(shap_fidelity_score_list):.4f}')
print(f'SHAP Std Fidelity_Score: {np.std(shap_fidelity_score_list):.4f}')

LIME Mean Fidelity_Score: 0.8828
LIME Std Fidelity_Score: 0.0757

SHAP Mean Fidelity_Score: 1.0000
SHAP Std Fidelity_Score: 0.0000


In [25]:
lime_predict_array = np.array([int(results['lime_explanations_list_local_pred'][i]>0.5) for i in range(results['len_x_test'])])
shap_predict_array = np.array([int(results['exp_shap_expected_value'][1] + np.sum(results['shap_values'][i][:, 1])>0.5) for i in range(results['len_x_test'])])
lime_predict_array[0], results['selected_model_predict_array'][0]

kronecker_delta = lambda x, y: [1 if a == b else 0 for a, b in zip(x, y)]
lime_fidelity_decision_list = kronecker_delta(lime_predict_array, results['selected_model_predict_array'])
print(f'LIME Mean Fidelity_Decision: {np.mean(lime_fidelity_decision_list):.4f}')
print(f'LIME Std Fidelity_Decision: {np.std(lime_fidelity_decision_list):.4f}')

shap_fidelity_decision_list = kronecker_delta(shap_predict_array, results['selected_model_predict_array'])
print('')
print(f'LIME Mean Fidelity_Decision: {np.mean(shap_fidelity_decision_list):.4f}')
print(f'LIME Std Fidelity_Decision: {np.std(shap_fidelity_decision_list):.4f}')

LIME Mean Fidelity_Decision: 0.9825
LIME Std Fidelity_Decision: 0.1313

LIME Mean Fidelity_Decision: 1.0000
LIME Std Fidelity_Decision: 0.0000


In [26]:
# Calculate simplicity
def calculate_simplicity(_values, threshold=0.05):
    simplicity_scores = []
    for instance_value in _values:
        abs_values = np.abs(instance_value)
        num_important_features = np.sum( abs_values > (max(abs_values) *threshold))
        simplicity_scores.append(num_important_features)
    return simplicity_scores


def get_simplicity_dict(name, values, threshold_list=[0.1, 0.05, 0.01], op='mean'):
    d = {'Method': name}
    for threshold in threshold_list:
        if op == 'mean':
            d[threshold] = np.mean(calculate_simplicity(values, threshold))
        elif op == 'std':
            d[threshold] = np.std(calculate_simplicity(values, threshold))
        else:
            raise('error')
    return d

lime_feature_values = [list(zip(*results['lime_explanations_list__'][i]))[1] for i in range(results['len_x_test'])]

list_of_scores_dict = [get_simplicity_dict(name, values, op='mean') for name, values in 
                       [('LIME', lime_feature_values), ('SHAP', results['shap_values'][:,:,0]), ('EBM', results['ebm_local_scores']),
                        ('TabNet', results['explain_tabnet_matrix'])]]

_df_simplicity = pd.DataFrame(list_of_scores_dict)
_df_simplicity.set_index('Method', inplace=True)
display('Mean of Simplicity')
display(_df_simplicity)


list_of_scores_dict = [get_simplicity_dict(name, values, op='std') for name, values in 
                       [('LIME', lime_feature_values), ('SHAP', results['shap_values'][:,:,0]), ('EBM', results['ebm_local_scores']),
                        ('TabNet', results['explain_tabnet_matrix'])]]

_df_simplicity = pd.DataFrame(list_of_scores_dict)
_df_simplicity.set_index('Method', inplace=True)
display('S.Dev of Simplicity')
_df_simplicity

'Mean of Simplicity'

,0.1,0.05,0.01
Method,,,
LIME,9.929825,10.000000,10.000000
SHAP,13.859649,17.421053,25.461988
EBM,28.403509,43.549708,56.076023
TabNet,6.210526,6.690058,7.368421


'S.Dev of Simplicity'

,0.1,0.05,0.01
Method,,,
LIME,0.368003,0.000000,0.000000
SHAP,2.440678,2.826032,2.340219
EBM,5.371115,5.196899,1.555995
TabNet,9.559424,9.734058,9.841965


In [27]:
# Consistency measures how similar the explanations produced by different XAI methods are when applied to the same input data

def retain_top_features(features, n=5):
    arr = np.abs(features)
    # Get the indices of the top n maximum values
    indices = np.argpartition(arr, -n)[-n:]

    # Create a mask
    mask = np.zeros_like(arr, dtype=bool)
    mask[indices] = True
    return np.where(mask, features, 0)

shap_val = lambda x, top_n=2: retain_top_features(results['shap_values'] [x][:, 1], top_n)
lime_val = lambda x, top_n=2: retain_top_features(get_lime_feature_importance(lime_explanations_list[x]), top_n)
lime_val = lambda x, top_n=2: retain_top_features(get_lime_feature_importance(results['lime_explanations_list__'][x]), top_n)
tabnet_val = lambda x, top_n=2: retain_top_features(results['explain_tabnet_matrix'][x], top_n)

def get_lime_feature_importance(lime_exp):
    feature_importance = {}
    for feature, sc in lime_exp:
        # print(feature)
        # Split feature by '<', '>', or '=' and take the first part as the feature name
        if feature.find('< ')>-1:
            feature = feature.split(' < ')[1]
        feature = feature.split(' > ')[0].strip()
        feature = feature.split(' <= ')[0].strip()
        for f in ['>', '<', '=']:
            if feature.find(f)>-1:
                print('error', feature)
        feature_importance[feature] = sc
    # print(feature_importance)
    # print([feature_importance.get(feature, 0) for feature in feature_names])
    return np.array([feature_importance.get(feature, 0) for feature in results['feature_names']])

from scipy.stats import spearmanr

def get_consistency(method1, method2):
    numerator = 0
    divider = 0
    avg_sp_corr_list = []
    for i in range(results['len_x_test']): 
        # Compute Spearman's rank correlation coefficient
        spearman_corr_array = np.array([spearmanr(method1(i, top_n=n), method2(i, top_n=n))[0] for n in range(1, 6)]) #only top 5 features: 1 feature is counted 5 five times, 2 four times,.. 5th only one time.
        avg_spearman_corr = spearman_corr_array.mean()
        avg_sp_corr_list.append(avg_spearman_corr)
        
    return np.array(avg_sp_corr_list)

lime_shap_consistency_list = get_consistency(lime_val, shap_val)
lime_tabnet_consistency_list = get_consistency(lime_val, tabnet_val)
tabnet_shap_consistency_list = get_consistency(tabnet_val, shap_val)

print('Consistency')

print('Mean')
print(lime_shap_consistency_list.shape)
print(f'LIME, SHAP Consistency: {lime_shap_consistency_list.mean():.2f}')
print(f'LIME, TabNet Consistency: {lime_tabnet_consistency_list.mean():.2f}')
print(f'TabNet, SHAP Consistency: {tabnet_shap_consistency_list.mean():.2f}')
print('')
print('S.Dev')
print(f'LIME, SHAP Consistency: {lime_shap_consistency_list.std():.2f}')
print(f'LIME, TabNet Consistency: {lime_tabnet_consistency_list.std():.2f}')
print(f'TabNet, SHAP Consistency: {tabnet_shap_consistency_list.std():.2f}')

Consistency
Mean
(171,)
LIME, SHAP Consistency: 0.58
LIME, TabNet Consistency: -0.05
TabNet, SHAP Consistency: -0.06

S.Dev
LIME, SHAP Consistency: 0.21
LIME, TabNet Consistency: 0.22
TabNet, SHAP Consistency: 0.20


In [28]:
# Function to calculate changes in explanations
def calculate_robustness(original, noisy):
    arr_org = np.array(original)
    arr_noisy = np.array(noisy)
    change_matrix = np.abs(arr_org - arr_noisy)
    divisor = max( [arr_org.max(), arr_noisy.max()] ) - min(  [arr_org.min(), arr_noisy.min()]  )  # max - min normalise
    robustness_list = change_matrix/divisor # divided in order to normalise between 0 to 1 (otherwise when the feature importance goes beyond 1 and under -1 such as in tabnet, ebm this won't normalise to 1)
    return np.mean(robustness_list)

def lime_robustness(original, noisy):
    means_list = []
    _max, _min = 0, 0
    for i in range(len(original)):
        # d1, d2 = dict(original[i].as_list()), dict(noisy[i].as_list())
        d1, d2 = dict(original[i]), dict(noisy[i])
        # print(d1.keys())
        # print(d2.keys())
        v1, v2 = list(zip(*[[d1.get(f, 0), d2.get(f, 0)] for f in set(list(d1.keys())+list(d2.keys()))]))
        arr_org = np.array(v1)
        arr_noisy = np.array(v2)
        _max = max( [arr_org.max(), arr_noisy.max(), _max] )
        _min = min( [arr_org.min(), arr_noisy.min(), _min] )
        means_list.append(np.abs(arr_org - arr_noisy).mean())
    divisor = _max - _min
    robustness_list = list(map(lambda x: x/divisor, means_list)) # max - min normalise
    return np.array(robustness_list)
        


lime_robustness_list = lime_robustness(results['lime_explanations_list__'], results['lime_explanations_list_noisy__'])
shap_robustness_list = calculate_robustness(results['shap_values'], results['shap_values_noisy'])
ebm_robustness_list = calculate_robustness(results['ebm_local_scores'], results['ebm_local_scores_noisy'])
tabnet_robustness_list = calculate_robustness(results['explain_tabnet_matrix'], results['explain_tabnet_matrix_noisy'])

print('Robustness')
print('Mean')
print(f"LIME: {lime_robustness_list.mean():.4f}")
print(f"SHAP: {shap_robustness_list.mean():.4f}")
print(f"EBM: {ebm_robustness_list.mean():.4f}")
print(f"TabNet: {tabnet_robustness_list.mean():.4f}")
print('')
print('S.Dev')
print(f"LIME: {lime_robustness_list.std():.4f}")
print(f"SHAP: {shap_robustness_list.std():.4f}")
print(f"EBM: {ebm_robustness_list.std():.4f}")
print(f"TabNet: {tabnet_robustness_list.std():.4f}")

Robustness
Mean
LIME: 0.0422
SHAP: 0.0078
EBM: 0.0175
TabNet: 0.0000

S.Dev
LIME: 0.0201
SHAP: 0.0000
EBM: 0.0000
TabNet: 0.0000


In [29]:
# Anchors
threshold_list = [0.9, 0.95, 0.99]
print('Mean')
for threshold in map(str, threshold_list):
    precison, coverage, simplicity, robustness = np.array(results['anchors_metrics_dict'][threshold]).mean(axis=0)
    print('Threshold at', threshold)
    print(f' Precison: {precison:.2f}')
    print(f' Coverage: {coverage:.2f}')
    print(f' Simplicity: {simplicity:.2f}')
    print(f' Robustness: {robustness:.2f}')

print('')
print('S.Dev')

for threshold in map(str, threshold_list):
    precison, coverage, simplicity, robustness = np.array(results['anchors_metrics_dict'][threshold]).std(axis=0)
    print('Threshold at', threshold)
    print(f' Precison: {precison:.2f}')
    print(f' Coverage: {coverage:.2f}')
    print(f' Simplicity: {simplicity:.2f}')
    print(f' Robustness: {robustness:.2f}')

Mean
Threshold at 0.9
 Precison: 0.99
 Coverage: 0.22
 Simplicity: 2.03
 Robustness: 0.72
Threshold at 0.95
 Precison: 0.99
 Coverage: 0.22
 Simplicity: 2.05
 Robustness: 0.72
Threshold at 0.99
 Precison: 1.00
 Coverage: 0.18
 Simplicity: 2.37
 Robustness: 0.70

S.Dev
Threshold at 0.9
 Precison: 0.02
 Coverage: 0.11
 Simplicity: 0.17
 Robustness: 0.18
Threshold at 0.95
 Precison: 0.01
 Coverage: 0.09
 Simplicity: 0.25
 Robustness: 0.21
Threshold at 0.99
 Precison: 0.00
 Coverage: 0.08
 Simplicity: 0.57
 Robustness: 0.22
